In [1]:
import cda2
import json
import re
import pyspark.sql.types as T
import pyspark.sql.functions as F
#import datetime
import time

from datetime import datetime, timedelta
from timeit import default_timer as timer
from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number

# Connect to Spark

In [2]:
api = cda2.Api(timeout=120)

Set configuration parameters to better optimize queries.

In [3]:
config = {
    "spark.sql.adaptive.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.parallelismFirst": "false",
    "spark.sql.adaptive.coalescePartitions.minPartitionSize": "1m",
    "spark.executor.memory": "16g",
    "spark.executor.memoryOverhead": "16g",
    "spark.reducer.maxReqsInFlight": "1",
    "spark.shuffle.io.retryWait": "60s",
    "spark.shuffle.io.maxRetries": "10",
}

# config = {
#     # "spark.sql.adaptive.enabled": "true",
#     # "spark.sql.adaptive.coalescePartitions.enabled": "true",
#     # "spark.sql.adaptive.coalescePartitions.parallelismFirst": "false",
#     # "spark.sql.adaptive.coalescePartitions.minPartitionSize": "1m",
#     "spark.executor.memory": "16g",
#     "spark.executor.memoryOverhead": "16g",
# }

Start Spark and specify number of cpus to use. 400 is quite high, but we'll be running 1 year at a time and want to have it done in just a few minutes.

In [4]:
#api.start_spark(n_executors=400, config=config)
api.start_spark(n_executors=400)

https://artifacts.mitre.org/artifactory/java-libs-release added as a remote repository with the name: repo-1
https://dali.mitre.org/nexus/content/repositories/mitre-caasd-releases added as a remote repository with the name: repo-2
https://dali.mitre.org/nexus/content/repositories/external-releases added as a remote repository with the name: repo-3
Ivy Default Cache set to: /home/rchong/.ivy2/cache
The jars for the packages stored in: /home/rchong/.ivy2/jars
org.mitre.spark#spark-geo_spark3.5_2.12 added as a dependency
org.apache.spark#spark-avro_2.12 added as a dependency
graphframes#graphframes added as a dependency
org.mongodb.spark#mongo-spark-connector_2.12 added as a dependency
com.oracle.database.jdbc#ojdbc8 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-4970bf0b-4fd7-4d27-9eff-a1c6dfbc231a;1.0
	confs: [default]


:: loading settings :: url = jar:file:/devel/data_access/software/tdp-jupyter/poetry/cache/virtualenvs/python39-QwwvzYkJ-py3.9/lib/python3.9/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found org.mitre.spark#spark-geo_spark3.5_2.12;0.2.0 in repo-1
	found org.mitre.spark#spark-geo-core_2.12;0.2.0 in repo-1
	found org.scala-lang.modules#scala-collection-compat_2.12;2.11.0 in central
	found net.sf.geographiclib#GeographicLib-Java;2.0 in central
	found org.ejml#ejml-core;0.43.1 in central
	found org.ejml#ejml-ddense;0.43.1 in central
	found com.esri.geometry#esri-geometry-api;2.2.4 in central
	found com.fasterxml.jackson.core#jackson-core;2.9.6 in central
	found com.google.geometry#s2-geometry;2.0.0 in central
	found com.google.guava#guava;25.1-jre in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found org.checkerframework#checker-qual;2.0.0 in central
	found com.google.errorprone#error_prone_annotations;2.1.3 in central
	found com.google.j2objc#j2objc-annotations;1.1 in central
	found org.codehaus.mojo#animal-sniffer-annotations;1.14 in central
	found com.uber#h3;4.1.1 in central
	found org.apache.spark#spark-avro_2.12;3.5.1 in central
	found org.tuka

In [5]:
api.spark.sparkContext.setLogLevel("ERROR")

In [6]:
year0 = "2025"
year1 = str(int(year0) + 1)

In [7]:
datelist = [
    year0 + "0101",
    year0 + "0201",
    year0 + "0301",
    year0 + "0401",
    year0 + "0501",
    year0 + "0601",
    year0 + "0701",
    year0 + "0801",
    year0 + "0901",
    year0 + "1001",
    year0 + "1101",
    year0 + "1201",
    year1 + "0101",
]

datelist = [
    year0 + "0101",
    year1 + "0101",
]

datelist = [
    year0 + "0101",
    year0 + "0401",
    year0 + "0701",
    year0 + "1001",
    year1 + "0101",
]

datelist = [
    year0 + "0101",
    year0 + "0301",
    year0 + "0501",
    year0 + "0701",
    year0 + "0901",
    year0 + "1101",
    year1 + "0101",
]

datelist = [
    year0 + "0101",
    year0 + "0401",
    year0 + "0701",
    year0 + "1001",
    year1 + "0101",
]

datelist = [
    year0 + "0901",
    year1 + "0101",
]

datelist = [
    year0 + "0101",
    year0 + "0501",
    year0 + "0901",
    year1 + "0101",
]


In [8]:
script = "retrieve_traffic_v4.ipynb"

In [9]:
i = 0
do_litetracks = True
do_flightplans = True
do_airspace_assignments = True

In [10]:
if i < len(datelist) - 1:
    dates = {"start_date": datelist[i], "end_date": datelist[i+1]}
    print("dates: ", dates)
    print("start time: ", datetime.now())
    %run $script
    i += 1
    print("end time: ", datetime.now())

dates:  {'start_date': '20250101', 'end_date': '20250501'}
start time:  2026-02-17 08:07:36.292320
starting:  {'start_date': '20250101', 'end_date': '20250501'}
   retrieving litetracks:  2026-02-17 08:07:41.441365


Multiple versions found: 3.1.71, 3.1.73, 3.1.74
Multiple versions found: 3.1.71, 3.1.73, 3.1.74                                 


   retrieving planned_routes:  2026-02-17 08:07:48.116708


Multiple versions found: 3.1.71, 3.1.73, 3.1.74
Multiple versions found: 3.1.71, 3.1.73, 3.1.74                                 


   retrieving flightplans:  2026-02-17 08:07:52.603392
   retrieving airspace_assignments:  2026-02-17 08:07:54.132689


...

      df_master size: 4039413 2026-02-17 08:08:57.100478
   started saving litetracks:  2026-02-17 08:08:57.859962


                                                                                 115]]6]]

   completed saving litetracks:  2026-02-17 08:11:05.296002
   started saving flightplans:  2026-02-17 08:11:05.302307


   completed saving flightplans:  2026-02-17 08:11:11.183629
   started saving airspace_assignments:  2026-02-17 08:11:11.188909


   completed saving airspace_assignments:  2026-02-17 08:11:23.563151
end time:  2026-02-17 08:11:23.563728


In [11]:
if i < len(datelist) - 1:
    dates = {"start_date": datelist[i], "end_date": datelist[i+1]}
    print("dates: ", dates)
    print("start time: ", datetime.now())
    %run $script
    i += 1
    print("end time: ", datetime.now())

dates:  {'start_date': '20250501', 'end_date': '20250901'}
start time:  2026-02-17 08:11:23.576420
starting:  {'start_date': '20250501', 'end_date': '20250901'}
   retrieving litetracks:  2026-02-17 08:11:23.585699


Multiple versions found: 3.1.74, 3.1.75, 3.1.76, 3.1.77
Multiple versions found: 3.1.74, 3.1.75, 3.1.76


   retrieving planned_routes:  2026-02-17 08:11:24.414593


Multiple versions found: 3.1.74, 3.1.75, 3.1.76, 3.1.77
Multiple versions found: 3.1.74, 3.1.75, 3.1.76, 3.1.77


   retrieving flightplans:  2026-02-17 08:11:25.431288
   retrieving airspace_assignments:  2026-02-17 08:11:26.153268


...

      df_master size: 4353836 2026-02-17 08:12:03.854944
   started saving litetracks:  2026-02-17 08:12:04.549803


                                                                                 118]]]]]]]]

   completed saving litetracks:  2026-02-17 08:14:50.855445
   started saving flightplans:  2026-02-17 08:14:50.858273


   completed saving flightplans:  2026-02-17 08:14:58.644064
   started saving airspace_assignments:  2026-02-17 08:14:58.649342


   completed saving airspace_assignments:  2026-02-17 08:15:15.062148
end time:  2026-02-17 08:15:15.062879


In [12]:
if i < len(datelist) - 1:
    dates = {"start_date": datelist[i], "end_date": datelist[i+1]}
    print("dates: ", dates)
    print("start time: ", datetime.now())
    %run $script
    i += 1
    print("end time: ", datetime.now())

dates:  {'start_date': '20250901', 'end_date': '20260101'}
start time:  2026-02-17 08:15:15.075639
starting:  {'start_date': '20250901', 'end_date': '20260101'}
   retrieving litetracks:  2026-02-17 08:15:15.083158


Multiple versions found: 3.1.77, 3.1.78, 3.1.79, 3.1.80
Multiple versions found: 3.1.76, 3.1.77, 3.1.79, 3.1.80


   retrieving planned_routes:  2026-02-17 08:15:15.931953


Multiple versions found: 3.1.77, 3.1.78, 3.1.79, 3.1.80
Multiple versions found: 3.1.77, 3.1.78, 3.1.79, 3.1.80


   retrieving flightplans:  2026-02-17 08:15:16.870422
   retrieving airspace_assignments:  2026-02-17 08:15:17.644363


...

      df_master size: 4128867 2026-02-17 08:16:04.943188
   started saving litetracks:  2026-02-17 08:16:05.604419


                                                                                 1600]]5]]]]

   completed saving litetracks:  2026-02-17 08:18:10.016797
   started saving flightplans:  2026-02-17 08:18:10.019269


   completed saving flightplans:  2026-02-17 08:18:17.124012
   started saving airspace_assignments:  2026-02-17 08:18:17.130526


   completed saving airspace_assignments:  2026-02-17 08:18:30.863220
end time:  2026-02-17 08:18:30.863965


In [13]:
if i < len(datelist) - 1:
    dates = {"start_date": datelist[i], "end_date": datelist[i+1]}
    print("dates: ", dates)
    print("start time: ", datetime.now())
    %run $script
    i += 1
    print("end time: ", datetime.now())

In [14]:
if i < len(datelist) - 1:
    dates = {"start_date": datelist[i], "end_date": datelist[i+1]}
    print("dates: ", dates)
    print("start time: ", datetime.now())
    %run $script
    i += 1
    print("end time: ", datetime.now())

In [15]:
if i < len(datelist) - 1:
    dates = {"start_date": datelist[i], "end_date": datelist[i+1]}
    print("dates: ", dates)
    print("start time: ", datetime.now())
    %run $script
    i += 1
    print("end time: ", datetime.now())

In [16]:
if i < len(datelist) - 1:
    dates = {"start_date": datelist[i], "end_date": datelist[i+1]}
    print("dates: ", dates)
    print("start time: ", datetime.now())
    %run $script
    i += 1
    print("end time: ", datetime.now())

In [17]:
if i < len(datelist) - 1:
    dates = {"start_date": datelist[i], "end_date": datelist[i+1]}
    print("dates: ", dates)
    print("start time: ", datetime.now())
    %run $script
    i += 1
    print("end time: ", datetime.now())

In [18]:
if i < len(datelist) - 1:
    dates = {"start_date": datelist[i], "end_date": datelist[i+1]}
    print("dates: ", dates)
    print("start time: ", datetime.now())
    %run $script
    i += 1
    print("end time: ", datetime.now())

In [19]:
if i < len(datelist) - 1:
    dates = {"start_date": datelist[i], "end_date": datelist[i+1]}
    print("dates: ", dates)
    print("start time: ", datetime.now())
    %run $script
    i += 1
    print("end time: ", datetime.now())

In [20]:
if i < len(datelist) - 1:
    dates = {"start_date": datelist[i], "end_date": datelist[i+1]}
    print("dates: ", dates)
    print("start time: ", datetime.now())
    %run $script
    i += 1
    print("end time: ", datetime.now())

In [21]:
if i < len(datelist) - 1:
    dates = {"start_date": datelist[i], "end_date": datelist[i+1]}
    print("dates: ", dates)
    print("start time: ", datetime.now())
    %run $script
    i += 1
    print("end time: ", datetime.now())